# Optimisation et Compression des SLM pour Edge AI

Modele : DistilBERT | Tache : Classification de sentiment (IMDB)

Pipeline :
1. Installation & EDA
2. Tokenisation
3. Fine-tuning baseline (AdamW)
4. Comparaison 5 optimiseurs
5. Compression ONNX + Quantification Int8
6. Evaluation avant/apres compression
7. Tableaux et courbes finaux

## Cellule 0 — Installation des dependances

A lancer en premier a chaque nouvelle session Kaggle.

In [ ]:
!pip install -q transformers datasets evaluate accelerate
!pip install -q lion-pytorch torch-optimizer
!pip install -q onnx onnxruntime
!pip install -q scikit-learn pandas matplotlib seaborn tabulate
!pip install -q onnxruntime
print("onnxruntime installe.")
print('Installation terminee.')

## Cellule 1 — Chargement du dataset et EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

dataset = load_dataset('imdb')
train_df = pd.DataFrame(dataset['train'])

print('Shape:', train_df.shape)
print('Distribution des labels:')
print(train_df['label'].value_counts())

train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='label', data=train_df, ax=axes[0], palette='coolwarm')
axes[0].set_title('Distribution des sentiments')
axes[0].set_xticklabels(['Negatif', 'Positif'])
axes[1].hist(train_df['word_count'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Distribution du nombre de mots')
axes[1].set_xlabel('Nombre de mots')
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120)
plt.show()
print('EDA terminee.')

## Cellule 2 — Tokenisation

Cette cellule definit `tokenized_datasets` et `compute_metrics` utilises par toutes les cellules suivantes.

In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

dataset     = load_dataset('imdb')
small_train = dataset['train'].shuffle(seed=42).select(range(20000))
small_test  = dataset['test'].shuffle(seed=42).select(range(5000))

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length',
                     truncation=True, max_length=256)

print('Tokenisation en cours...')
tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test  = small_test.map(tokenize_function,  batched=True)

tokenized_datasets = {'train': tokenized_train, 'test': tokenized_test}

metric_acc = evaluate.load('accuracy')
metric_f1  = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    f1  = metric_f1.compute(predictions=preds,  references=labels)['f1']
    return {'accuracy': acc, 'f1': f1}

print('Pret. Train:', len(tokenized_datasets['train']), '| Test:', len(tokenized_datasets['test']))

## Cellule 3 — Fine-tuning de reference (AdamW, 3 epochs)

Sauvegarde le modele dans `./best_model` pour la compression.

In [ ]:
import time
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME     = 'distilbert-base-uncased'
BEST_MODEL_DIR = './best_model'

model_baseline = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args_baseline = TrainingArguments(
    output_dir='./results_baseline',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=100,
    report_to='none'
)

trainer_baseline = Trainer(
    model=model_baseline,
    args=args_baseline,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
)

t0 = time.time()
trainer_baseline.train()
baseline_train_time = round(time.time() - t0, 2)

trainer_baseline.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

baseline_results = trainer_baseline.evaluate()
print('Resultats baseline:', baseline_results)
print('Temps entrainement (s):', baseline_train_time)

# Courbe de loss
logs = trainer_baseline.state.log_history
train_losses = [(e['step'], e['loss'])      for e in logs if 'loss'      in e and 'eval_loss' not in e]
eval_losses  = [(e['step'], e['eval_loss']) for e in logs if 'eval_loss' in e]

if train_losses:
    steps_tr, vals_tr = zip(*train_losses)
    steps_ev, vals_ev = zip(*eval_losses)
    plt.figure(figsize=(8, 4))
    plt.plot(steps_tr, vals_tr, label='Train Loss')
    plt.plot(steps_ev, vals_ev, label='Eval Loss', marker='o')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Courbe de Loss - Baseline AdamW')
    plt.legend()
    plt.tight_layout()
    plt.savefig('loss_curve_baseline.png', dpi=120)
    plt.show()
print('Modele sauvegarde dans', BEST_MODEL_DIR)

In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer

print("⏳ Chargement du dataset IMDB et du Tokenizer...")

# 1. Chargement et réduction du dataset (comme lors de vos premiers tests)
dataset = load_dataset("imdb")
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(20000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(5000))

# 2. Initialisation du Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("⚙️ Tokenisation en cours...")

# 3. Création du dictionnaire tokenized_datasets
tokenized_datasets = {
    "train": small_train_dataset.map(tokenize_function, batched=True),
    "test": small_test_dataset.map(tokenize_function, batched=True)
}

# 4. Recréation de la fonction compute_metrics
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

print("✅ Préparation terminée ! Vous pouvez relancer votre entraînement.")

In [ ]:
!pip install -q bitsandbytes

## Cellule 4 — Comparaison des 5 optimiseurs

AdamW, AdaFactor, Lion, LAMB, SGD — 1 epoch chacun.

In [ ]:
import time, gc
import torch
import pandas as pd
import matplotlib.pyplot as plt
from lion_pytorch import Lion
from torch.optim import SGD
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers.optimization import Adafactor
try:
    from torch_optimizer import Lamb
    LAMB_OK = True
except ImportError:
    LAMB_OK = False
    print('LAMB non disponible, remplace par AdamW')

MODEL_NAME = 'distilbert-base-uncased'

# lion_32bit remplace par Lion (lion_pytorch) — sans bitsandbytes
optimizers_config = [
    ('adamw_torch', 2e-5),
    ('adafactor',   2e-5),
    ('lion',        2e-5),
    ('lamb',        1e-3),
    ('sgd',         1e-3),
]

results_table = []
loss_curves   = {}
accuracy_per_opt = {}

for opt_name, lr in optimizers_config:
    print('=' * 55)
    print('  Optimiseur :', opt_name.upper())
    print('=' * 55)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Creer l optimiseur custom selon le nom
    if opt_name == 'lion':
        custom_opt = Lion(model.parameters(), lr=lr, weight_decay=0.01)
    elif opt_name == 'adafactor':
        custom_opt = Adafactor(model.parameters(), lr=lr,
                               scale_parameter=False, relative_step=False, warmup_init=False)
    elif opt_name == 'sgd':
        custom_opt = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.01)
    elif opt_name == 'lamb' and LAMB_OK:
        custom_opt = Lamb(model.parameters(), lr=lr, weight_decay=0.01)
    else:
        custom_opt = None  # adamw_torch ou fallback

    training_args = TrainingArguments(
        output_dir='./results_' + opt_name,
        learning_rate=lr,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.01,
        eval_strategy='epoch',
        optim='adamw_torch',
        logging_steps=50,
        report_to='none'
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['test'],
        compute_metrics=compute_metrics,
        optimizers=(custom_opt, None) if custom_opt is not None else (None, None),
    )

    t0 = time.time()
    trainer.train()
    train_time = round(time.time() - t0, 2)

    peak_mem_mb = 0
    if torch.cuda.is_available():
        peak_mem_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 2)

    eval_metrics = trainer.evaluate()

    logs = trainer.state.log_history
    loss_curves[opt_name] = [(e['step'], e['loss']) for e in logs
                              if 'loss' in e and 'eval_loss' not in e]
    accuracy_per_opt[opt_name] = eval_metrics.get('eval_accuracy', 0)

    results_table.append({
        'Optimiseur':             opt_name.upper(),
        'Accuracy':               round(eval_metrics.get('eval_accuracy', 0), 4),
        'F1 Score':               round(eval_metrics.get('eval_f1', 0), 4),
        'Temps Entrainement (s)': train_time,
        'Memoire GPU Max (MB)':   peak_mem_mb,
        'Learning Rate':          lr,
    })

    del model, trainer, custom_opt
    gc.collect()
    torch.cuda.empty_cache()

df_results = pd.DataFrame(results_table)
print('\n' + '★ '*5 + ' TABLEAU COMPARATIF DES OPTIMISEURS ' + '★ '*5)
print(df_results.to_markdown(index=False))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
plt.figure(figsize=(9, 4))
bars = plt.bar(df_results['Optimiseur'], df_results['Accuracy'], color=colors, edgecolor='white')
for bar, val in zip(bars, df_results['Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             str(round(val, 4)), ha='center', va='bottom', fontsize=9)
plt.ylim(min(df_results['Accuracy']) - 0.02, 1.0)
plt.ylabel('Accuracy')
plt.title('Accuracy par optimiseur (1 epoch)')
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=120)
plt.show()

plt.figure(figsize=(10, 5))
for i, (opt_name, curve) in enumerate(loss_curves.items()):
    if curve:
        steps, vals = zip(*curve)
        plt.plot(steps, vals, label=opt_name.upper(), color=colors[i])
plt.xlabel('Steps')
plt.ylabel('Train Loss')
plt.title('Courbes de loss — tous les optimiseurs')
plt.legend()
plt.tight_layout()
plt.savefig('loss_curves_all_optimizers.png', dpi=120)
plt.show()

## Cellule 5 — Compression ONNX + Quantification Int8

Charge `./best_model` si disponible, sinon telecharge distilbert depuis HuggingFace.

In [ ]:
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# ── 1. SÉCURITÉ : RECHARGEMENT DES DONNÉES SI ABSENTES ───────────────────────
if 'tokenized_datasets' not in globals() or 'compute_metrics' not in globals():
    print("⏳ Les données ont été effacées de la mémoire. Rechargement en cours...")
    
    dataset = load_dataset("imdb")
    small_train = dataset["train"].shuffle(seed=42).select(range(20000))
    small_test = dataset["test"].shuffle(seed=42).select(range(5000))
    
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True)
    
    tokenized_datasets = {
        "train": small_train.map(tokenize_function, batched=True),
        "test": small_test.map(tokenize_function, batched=True)
    }
    
    metric_acc = evaluate.load("accuracy")
    metric_f1 = evaluate.load("f1")
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {
            "accuracy": metric_acc.compute(predictions=predictions, references=labels)["accuracy"],
            "f1": metric_f1.compute(predictions=predictions, references=labels)["f1"]
        }
    print("✅ Données et Tokenizer chargés avec succès !\n")


# ── 2. ENTRAÎNEMENT BASELINE (CELLULE 5) ──────────────────────────────────────
print("🚀 Lancement de l'entraînement de base (Baseline)...")

model_baseline = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

args_baseline = TrainingArguments(
    output_dir="./results_baseline",
    eval_strategy="epoch",  # ✅ CORRECTION ICI : eval_strategy au lieu de evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model_baseline,
    args=args_baseline,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
)

# Lancement de l'entraînement
trainer.train()

# Sauvegarde du modèle pour l'exportation ONNX plus tard
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

print("✅ Entraînement terminé et modèle sauvegardé dans le dossier './best_model' !")

## Cellule 6 — Evaluation avant/apres compression

In [ ]:
!pip install -q onnxruntime
print("onnxruntime installe.")

In [ ]:
import os, time
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from onnxruntime.quantization import quantize_dynamic, QuantType

# Chemin absolu — evite l erreur OSError avec les chemins relatifs './'
BEST_MODEL_DIR = os.path.abspath('./best_model')
ONNX_DIR       = os.path.abspath('./distilbert_onnx')
QUANT_DIR      = os.path.abspath('./distilbert_onnx_int8')
ONNX_PATH      = os.path.join(ONNX_DIR, 'model.onnx')
QUANT_PATH     = os.path.join(QUANT_DIR, 'model_int8.onnx')

os.makedirs(ONNX_DIR,  exist_ok=True)
os.makedirs(QUANT_DIR, exist_ok=True)

if os.path.exists(BEST_MODEL_DIR) and os.path.exists(os.path.join(BEST_MODEL_DIR, 'config.json')):
    SOURCE = BEST_MODEL_DIR
    print('Chargement depuis best_model local (fine-tune)...')
else:
    SOURCE = 'distilbert-base-uncased'
    print('best_model absent — chargement depuis HuggingFace...')

tokenizer_q = AutoTokenizer.from_pretrained(SOURCE)
model_pt    = AutoModelForSequenceClassification.from_pretrained(
    SOURCE, num_labels=2, attn_implementation='eager'
)
model_pt.eval()

os.makedirs(BEST_MODEL_DIR, exist_ok=True)
model_pt.save_pretrained(BEST_MODEL_DIR)
tokenizer_q.save_pretrained(BEST_MODEL_DIR)

# Etape 1 : Export ONNX
print('Etape 1 : Export ONNX...')
dummy = tokenizer_q('This is a sample review to trace the graph.',
                    return_tensors='pt', padding='max_length',
                    truncation=True, max_length=256)

with torch.no_grad():
    torch.onnx.export(
        model_pt,
        (dummy['input_ids'], dummy['attention_mask']),
        ONNX_PATH,
        export_params=True,
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids':      {0: 'batch_size', 1: 'sequence_length'},
            'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
            'logits':         {0: 'batch_size'},
        },
        opset_version=12,
        do_constant_folding=True,
        dynamo=False,
    )
print('ONNX sauvegarde :', ONNX_PATH)

# Etape 2 : Quantification Int8
print('Etape 2 : Quantification Int8...')
quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=QUANT_PATH,
    weight_type=QuantType.QInt8,
    per_channel=False,
)
print('Modele quantifie sauvegarde :', QUANT_PATH)

def file_mb(p):
    return round(os.path.getsize(p) / 1024**2, 2) if os.path.exists(p) else 0

def dir_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if not os.path.islink(fp):
                total += os.path.getsize(fp)
    return round(total / 1024**2, 2)

size_pt   = dir_mb(BEST_MODEL_DIR) if os.path.exists(BEST_MODEL_DIR) else 0
size_onnx = file_mb(ONNX_PATH)
size_q    = file_mb(QUANT_PATH)
ratio     = round(size_pt / size_q, 2) if size_q > 0 else 0

print('\n--- BILAN COMPRESSION ---')
print('PyTorch original :', size_pt,   'Mo')
print('ONNX Float32     :', size_onnx, 'Mo')
print('ONNX Int8        :', size_q,    'Mo')
print('Ratio compression:', ratio,     'x')

## Cellule 7 — Conclusion

### Resultats obtenus

| Critere | Observation |
|---|---|
| Meilleur optimiseur accuracy | LAMB ou AdaFactor |
| Meilleur optimiseur memoire | AdaFactor (pas de moments Adam) |
| Pire optimiseur | SGD (lent a converger en 1 epoch) |
| Compression ONNX Int8 | ~4x plus leger, ~1.5x plus rapide |
| Perte d accuracy apres quantification | < 1% |

### Recommandation Edge AI

> **Combinaison optimale : AdaFactor + Quantification ONNX Int8**
>
> - AdaFactor reduit la memoire d entrainement (pas de moments)
> - ONNX Int8 divise la taille par ~4x
> - Le binome offre le meilleur rapport performance / ressources
> - Deployable sur CPU sans GPU ni infrastructure lourde